In [1]:
# ============================================================
# PROYECTO: Mercado Laboral y Pobreza en Ecuador 2007–2024
# Notebook 01: Obtención de Datos
# Autor: Bryan Anthony López Guerrero
# ============================================================
#
# Este notebook documenta las fuentes de datos utilizadas,
# las decisiones metodológicas de recolección y la estructura
# del pipeline de datos del proyecto.
#
# El procesamiento real de los datos se realiza en:
# → 02_obtencion_y_limpieza_datos.ipynb

print("=" * 60)
print("PROYECTO: Mercado Laboral y Pobreza en Ecuador 2007–2024")
print("Notebook 01 — Documentación de fuentes de datos")
print("=" * 60)

PROYECTO: Mercado Laboral y Pobreza en Ecuador 2007–2024
Notebook 01 — Documentación de fuentes de datos


In [2]:
fuentes = {
    "INEC – ENEMDU": {
        "descripcion": (
            "Encuesta Nacional de Empleo, Desempleo y Subempleo. "
            "Fuente oficial del gobierno ecuatoriano para estadísticas "
            "de mercado laboral, pobreza e ingresos."
        ),
        "url": "https://www.ecuadorencifras.gob.ec/estadisticas-laborales-enemdu/",
        "formato": "Boletines técnicos PDF + microdatos SPSS (.sav)",
        "periodo": "2007 – 2024 (diciembre de cada año)",
        "indicadores": [
            "Tasa de pobreza por ingresos (nacional, urbana, rural)",
            "Tasa de pobreza extrema",
            "Coeficiente de Gini",
            "Tasa de informalidad laboral",
            "Tasa de empleo adecuado",
            "Tasa de desempleo",
            "Ingreso laboral promedio mensual (USD)",
            "Línea de pobreza mensual (USD)",
        ],
        "licencia": "Creative Commons Atribución 4.0 Internacional",
    },
    "Banco Mundial": {
        "descripcion": (
            "Base de datos de indicadores de desarrollo mundial. "
            "Acceso gratuito vía API REST sin autenticación."
        ),
        "url": "https://datos.bancomundial.org",
        "api_url": "https://api.worldbank.org/v2/country/EC/indicator/",
        "formato": "JSON vía API REST",
        "periodo": "2000 – 2024",
        "indicadores": [
            "PIB per cápita en USD corrientes (NY.GDP.PCAP.CD)",
            "Crecimiento del PIB % (NY.GDP.MKTP.KD.ZG)",
            "Tasa de desempleo BM % (SL.UEM.TOTL.ZS)",
            "Empleo vulnerable % (SL.EMP.VULN.ZS)",
            "Empleo en agricultura % (SL.AGR.EMPL.ZS)",
            "Empleo en industria % (SL.IND.EMPL.ZS)",
            "Empleo en servicios % (SL.SRV.EMPL.ZS)",
            "Coeficiente Gini BM (SI.POV.GINI)",
        ],
        "licencia": "Creative Commons Atribución 4.0 Internacional",
    },
    "Banco Central del Ecuador (BCE)": {
        "descripcion": (
            "Institución emisora del Ecuador. Publica series históricas "
            "de indicadores macroeconómicos y laborales."
        ),
        "url": "https://estadisticas.bce.fin.ec",
        "formato": "Tablas manuales publicadas en boletines anuales",
        "periodo": "2007 – 2024",
        "indicadores": [
            "Salario básico unificado histórico (USD)",
        ],
        "licencia": "Datos públicos del Estado ecuatoriano",
    },
}

print("FUENTES DE DATOS DEL PROYECTO\n")
for nombre, info in fuentes.items():
    print(f"{'─'*55}")
    print(f"📌 {nombre}")
    print(f"   Descripción : {info['descripcion'][:80]}...")
    print(f"   URL         : {info['url']}")
    print(f"   Formato     : {info['formato']}")
    print(f"   Período     : {info['periodo']}")
    print(f"   Indicadores : {len(info['indicadores'])} variables")
    print(f"   Licencia    : {info['licencia']}")
print(f"{'─'*55}")

FUENTES DE DATOS DEL PROYECTO

───────────────────────────────────────────────────────
📌 INEC – ENEMDU
   Descripción : Encuesta Nacional de Empleo, Desempleo y Subempleo. Fuente oficial del gobierno ...
   URL         : https://www.ecuadorencifras.gob.ec/estadisticas-laborales-enemdu/
   Formato     : Boletines técnicos PDF + microdatos SPSS (.sav)
   Período     : 2007 – 2024 (diciembre de cada año)
   Indicadores : 8 variables
   Licencia    : Creative Commons Atribución 4.0 Internacional
───────────────────────────────────────────────────────
📌 Banco Mundial
   Descripción : Base de datos de indicadores de desarrollo mundial. Acceso gratuito vía API REST...
   URL         : https://datos.bancomundial.org
   Formato     : JSON vía API REST
   Período     : 2000 – 2024
   Indicadores : 8 variables
   Licencia    : Creative Commons Atribución 4.0 Internacional
───────────────────────────────────────────────────────
📌 Banco Central del Ecuador (BCE)
   Descripción : Institución emisora

In [3]:
import pandas as pd

# Tabla resumen de todos los indicadores del proyecto
indicadores = {
    'Variable en el dataset':  [
        'pobreza_ingresos_pct',
        'pobreza_extrema_pct',
        'pobreza_urbana_pct',
        'pobreza_rural_pct',
        'gini_inec',
        'desempleo_pct',
        'informalidad_pct',
        'empleo_adecuado_pct',
        'ingreso_laboral_promedio',
        'linea_pobreza_usd',
        'salario_basico_usd',
        'ratio_ingreso_pobreza',
        'pib_per_capita_usd',
        'crecimiento_pib_pct',
    ],
    'Descripción': [
        'Tasa de pobreza por ingresos nacional (%)',
        'Tasa de pobreza extrema nacional (%)',
        'Tasa de pobreza zona urbana (%)',
        'Tasa de pobreza zona rural (%)',
        'Coeficiente de Gini (desigualdad de ingresos)',
        'Tasa de desempleo nacional (%)',
        'Tasa de informalidad laboral (%)',
        'Tasa de empleo adecuado/pleno (%)',
        'Ingreso laboral promedio mensual (USD)',
        'Línea de pobreza mensual (USD)',
        'Salario básico unificado (USD)',
        'Ratio ingreso laboral / línea de pobreza',
        'PIB per cápita en USD corrientes',
        'Crecimiento del PIB (%)',
    ],
    'Fuente': [
        'INEC-ENEMDU', 'INEC-ENEMDU', 'INEC-ENEMDU', 'INEC-ENEMDU',
        'INEC-ENEMDU', 'INEC-ENEMDU', 'INEC-ENEMDU', 'INEC-ENEMDU',
        'INEC-ENEMDU', 'INEC-ENEMDU',
        'BCE',
        'Calculada',
        'Banco Mundial', 'Banco Mundial',
    ],
    'Período disponible': [
        '2007–2024', '2007–2024', '2007–2024', '2007–2024',
        '2007–2024', '2007–2024', '2011–2024', '2011–2024',
        '2011–2024', '2007–2024',
        '2007–2024',
        '2011–2024',
        '2000–2024', '2000–2024',
    ],
}

df_ind = pd.DataFrame(indicadores)
print("VARIABLES DEL DATASET MAESTRO\n")
print(df_ind.to_string(index=False))
print(f"\nTotal: {len(df_ind)} variables de 3 fuentes oficiales")

VARIABLES DEL DATASET MAESTRO

  Variable en el dataset                                   Descripción        Fuente Período disponible
    pobreza_ingresos_pct     Tasa de pobreza por ingresos nacional (%)   INEC-ENEMDU          2007–2024
     pobreza_extrema_pct          Tasa de pobreza extrema nacional (%)   INEC-ENEMDU          2007–2024
      pobreza_urbana_pct               Tasa de pobreza zona urbana (%)   INEC-ENEMDU          2007–2024
       pobreza_rural_pct                Tasa de pobreza zona rural (%)   INEC-ENEMDU          2007–2024
               gini_inec Coeficiente de Gini (desigualdad de ingresos)   INEC-ENEMDU          2007–2024
           desempleo_pct                Tasa de desempleo nacional (%)   INEC-ENEMDU          2007–2024
        informalidad_pct              Tasa de informalidad laboral (%)   INEC-ENEMDU          2011–2024
     empleo_adecuado_pct             Tasa de empleo adecuado/pleno (%)   INEC-ENEMDU          2011–2024
ingreso_laboral_promedio        I

In [4]:
print("=" * 60)
print("DECISIONES METODOLÓGICAS DE RECOLECCIÓN")
print("=" * 60)

decisiones = [
    {
        "decisión": "Usar diciembre de cada año como período de referencia",
        "razón": (
            "Los boletines anuales del INEC toman diciembre como mes de "
            "cierre para garantizar comparabilidad histórica. Usar otros "
            "meses generaría inconsistencias en la serie temporal."
        ),
    },
    {
        "decisión": "Construir el dataset INEC manualmente desde boletines PDF",
        "razón": (
            "Los microdatos ENEMDU (.sav) no estaban disponibles para "
            "descarga directa al momento del análisis. Los boletines "
            "técnicos oficiales son igualmente válidos para indicadores "
            "agregados nacionales. Los valores fueron verificados con "
            "múltiples fuentes secundarias."
        ),
    },
    {
        "decisión": "Obtener datos del Banco Mundial vía API REST directa",
        "razón": (
            "La librería wbgapi presentó errores de decodificación JSON "
            "al consultar múltiples años con separador de punto y coma. "
            "La API REST nativa de la URL "
            "api.worldbank.org/v2/country/EC/indicator/ "
            "es más estable y no requiere autenticación."
        ),
    },
    {
        "decisión": "Período de análisis 2007–2024 (comparabilidad desde 2007)",
        "razón": (
            "Según la guía oficial del INEC, la información estadística "
            "de la ENEMDU es comparable a partir de junio de 2007. "
            "Usar datos anteriores comprometería la validez comparativa "
            "de la serie histórica."
        ),
    },
    {
        "decisión": "Variables de informalidad disponibles solo desde 2011",
        "razón": (
            "La variable de tasa de informalidad no está disponible de "
            "forma comparable en los boletines ENEMDU antes de 2011. "
            "Esto limita el dataset del modelo ML a 14 observaciones "
            "(2011–2024), lo cual justifica el uso de LOOCV."
        ),
    },
    {
        "decisión": "No imputar valores nulos de informalidad 2007–2010",
        "razón": (
            "Imputar valores no observados con interpolación lineal "
            "introduciría suposiciones no verificables. Se prefiere "
            "reportar honestamente el período con datos disponibles."
        ),
    },
]

for i, d in enumerate(decisiones, 1):
    print(f"\n{i}. {d['decisión']}")
    print(f"   → {d['razón']}")

print("\n" + "=" * 60)

DECISIONES METODOLÓGICAS DE RECOLECCIÓN

1. Usar diciembre de cada año como período de referencia
   → Los boletines anuales del INEC toman diciembre como mes de cierre para garantizar comparabilidad histórica. Usar otros meses generaría inconsistencias en la serie temporal.

2. Construir el dataset INEC manualmente desde boletines PDF
   → Los microdatos ENEMDU (.sav) no estaban disponibles para descarga directa al momento del análisis. Los boletines técnicos oficiales son igualmente válidos para indicadores agregados nacionales. Los valores fueron verificados con múltiples fuentes secundarias.

3. Obtener datos del Banco Mundial vía API REST directa
   → La librería wbgapi presentó errores de decodificación JSON al consultar múltiples años con separador de punto y coma. La API REST nativa de la URL api.worldbank.org/v2/country/EC/indicator/ es más estable y no requiere autenticación.

4. Período de análisis 2007–2024 (comparabilidad desde 2007)
   → Según la guía oficial del INEC, la

In [5]:
print("=" * 60)
print("LIMITACIONES Y CONSIDERACIONES")
print("=" * 60)

limitaciones = [
    "Los datos son de fuente secundaria (INEC, BM, BCE). "
    "No se realizó levantamiento propio de información.",

    "El INEC realizó cambios metodológicos en la ENEMDU entre "
    "2020 y 2021 relacionados con la pandemia. Estos cambios "
    "afectan la comparabilidad estricta de ese período, aunque "
    "el INEC recalculó los factores de ponderación para "
    "mantener la comparabilidad de las cifras.",

    "El dataset del modelo ML tiene N=14 observaciones (años), "
    "lo que limita la complejidad de los modelos aplicables "
    "y requiere técnicas de validación especiales (LOOCV).",

    "Las proyecciones 2025 son estimaciones basadas en supuestos "
    "sobre variables macroeconómicas. No incorporan shocks "
    "externos imprevistos (crisis, desastres naturales, etc.).",

    "La variable de informalidad solo está disponible desde 2011, "
    "lo que reduce el período de análisis del modelo ML.",
]

for i, lim in enumerate(limitaciones, 1):
    print(f"\n{i}. {lim}")

print("\n" + "=" * 60)
print("Continúa en: 02_obtencion_y_limpieza_datos.ipynb")
print("=" * 60)

LIMITACIONES Y CONSIDERACIONES

1. Los datos son de fuente secundaria (INEC, BM, BCE). No se realizó levantamiento propio de información.

2. El INEC realizó cambios metodológicos en la ENEMDU entre 2020 y 2021 relacionados con la pandemia. Estos cambios afectan la comparabilidad estricta de ese período, aunque el INEC recalculó los factores de ponderación para mantener la comparabilidad de las cifras.

3. El dataset del modelo ML tiene N=14 observaciones (años), lo que limita la complejidad de los modelos aplicables y requiere técnicas de validación especiales (LOOCV).

4. Las proyecciones 2025 son estimaciones basadas en supuestos sobre variables macroeconómicas. No incorporan shocks externos imprevistos (crisis, desastres naturales, etc.).

5. La variable de informalidad solo está disponible desde 2011, lo que reduce el período de análisis del modelo ML.

Continúa en: 02_obtencion_y_limpieza_datos.ipynb
